# 01 — Limpieza y Preparacion de Datos

**Objetivo:** Cargar el dataset crudo de nacimientos DANE 2018, diagnosticar la calidad de los datos, aplicar el pipeline de limpieza y guardar el resultado procesado.

## Nota sobre la codificacion de PESO_NAC
El DANE utiliza **codigos categoricos** para PESO_NAC, no gramos reales:
| Codigo | Rango de peso |
|--------|---------------|
| 1 | < 1 000 g (bajo peso extremo) |
| 2 | 1 000 – 1 499 g (muy bajo peso) |
| 3 | 1 500 – 1 999 g (bajo peso) |
| 4 | 2 000 – 2 499 g (bajo peso) |
| 5 | 2 500 – 2 999 g (normal) |
| 6 | 3 000 – 3 499 g (normal) |
| 7 | 3 500 – 3 999 g (normal) |
| 8 | 4 000 – 4 499 g (grande) |
| 9 | >= 4 500 g / Sin info |

El filtro anterior `between(400, 7000)` era incorrecto — eliminaba **todos** los registros.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

# Asegurar que src/ este en el path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RAW_PATH = PROJECT_ROOT / "data" / "raw" / "nac2018.csv"
PROCESSED_PATH = PROJECT_ROOT / "data" / "processed" / "nac2018_cleaned.csv"

print(f"Directorio del proyecto: {PROJECT_ROOT}")
print(f"Archivo crudo: {RAW_PATH.exists()} — {RAW_PATH}")

## 1. Carga del dataset crudo

In [ ]:
from src.preprocessing import load_raw_data

df_raw = load_raw_data(RAW_PATH)
print(f"\nColumnas ({df_raw.shape[1]}):")
print(list(df_raw.columns))

## 2. Inspeccion inicial

In [ ]:
df_raw.head()

In [ ]:
df_raw.dtypes

In [ ]:
# Distribucion de PESO_NAC — confirmar que es categorico (1-9)
print("Distribucion de PESO_NAC (debe ser rango 1-9):")
print(df_raw["PESO_NAC"].value_counts().sort_index())

## 3. Reporte de valores faltantes

In [ ]:
null_counts = df_raw.isnull().sum()
null_pct = (null_counts / len(df_raw) * 100).round(2)

missing = (
    pd.DataFrame({"Nulos": null_counts, "Porcentaje (%)": null_pct})
    .sort_values("Nulos", ascending=False)
)
missing[missing["Nulos"] > 0]

## 4. Pipeline de limpieza

In [ ]:
from src.preprocessing import clean_data

df_clean = clean_data(df_raw)
print(f"\nShape final: {df_clean.shape}")
df_clean.head()

## 5. Ingenieria de variables

In [ ]:
from src.preprocessing import engineer_features

df_final = engineer_features(df_clean)

print("Distribucion de targets:")
print(f"  CESAREA   — {df_final['CESAREA'].mean():.1%} positivos ({df_final['CESAREA'].sum():,} casos)")
print(f"  BAJO_PESO — {df_final['BAJO_PESO'].mean():.1%} positivos ({df_final['BAJO_PESO'].sum():,} casos)")

In [ ]:
# Verificar que no quedan registros invalidos
print("Valores nulos en columnas clave:")
from src.preprocessing import FEATURES
print(df_final[FEATURES].isnull().sum())

## 6. Guardar dataset procesado

In [ ]:
PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)
df_final.to_csv(PROCESSED_PATH, index=False, encoding="utf-8")
print(f"Dataset guardado: {PROCESSED_PATH}")
print(f"Filas: {len(df_final):,}  |  Columnas: {df_final.shape[1]}")